In [ ]:
# Week 2: Text Cleaning and Normalization

## Overview:
Build a text cleaning pipeline to standardize MLS remarks and queries. Handle abbreviations (br → bedroom), measurements (2,000 sqft → 2000 square feet), prices (450k → 450000), unicode issues, and HTML remnants.

In [2]:
import pandas as pd
import re

df = pd.read_csv("../data/processed/listing_sample.csv")

In [4]:
df.shape

(1000, 7)

In [6]:
remarks = df["remarks"]

profile = {
    "row_count": len(df),
    "null_count": remarks.isna().sum(),
    "null_rate": remarks.isna().mean(),
    "average_length": remarks.str.len().mean(),
    "minimum_length": remarks.str.len().min(),
    "maximum_length": remarks.str.len().max(),
    "html_count": remarks.str.contains(
        r"<[^>]+>",
        regex=True,
        na=False
    ).sum(),
    "price_mentions": remarks.str.contains(
        r"\$\s*\d",
        regex=True,
        na=False
    ).sum(),
    "measurement_mentions": remarks.str.contains(
        r"\b(?:sqft|sq\.?\s*ft\.?)\b",
        case=False,
        regex=True,
        na=False
    ).sum(),
    "repeated_whitespace": remarks.str.contains(
        r"\s{2,}",
        regex=True,
        na=False
    ).sum()
}

profile

{'row_count': 1000,
 'null_count': np.int64(0),
 'null_rate': np.float64(0.0),
 'average_length': np.float64(1306.865),
 'minimum_length': np.int64(82),
 'maximum_length': np.int64(4000),
 'html_count': np.int64(0),
 'price_mentions': np.int64(48),
 'measurement_mentions': np.int64(172),
 'repeated_whitespace': np.int64(502)}

## Repeated whitespace
This initial profile reveals that there are 502 instances of repeated whitespace, now lets determine which type: tab, spaces, or line breaks.

In [7]:
whitespace_examples = df[
    remarks.str.contains(
        r"\s{2,}",
        regex=True,
        na=False
    )
]["remarks"].head(5)

for number, text in enumerate(whitespace_examples, start=1):
    print(f"EXAMPLE {number}")
    print(repr(text[:500]))
    print()

EXAMPLE 1
'Move in Ready Model!   Welcome to the Holly Plan 3, a beautifully upgraded single-story home featuring a fully landscaped backyard and over $188,000 in upgrades throughout. Designed for both comfort and functionality, this home offers 3 bedrooms, 2 bathrooms, a spacious 2-bay garage, flexible den, and a unique storage space with endless possibilities for a home office, reading nook, or additional flex space. The thoughtfully designed interior features luxury vinyl plank flooring throughout, char'

EXAMPLE 2
'Spacious 4-Bedroom Condo with Exceptional Living Space. This rarely available and uniquely large 4-bedroom condo offers an exceptional combination of space, comfort, and value. Designed for both everyday living and entertaining, this home features dual-pane windows that enhance energy efficiency while helping create a quieter indoor environment. The thoughtfully designed kitchen is highlighted by an elegant coffered ceiling, adding character and charm to the heart of the

All misc whitespace characters will be converted to spaces.

In [8]:
from collections import Counter

unicode_characters = Counter(
    character
    for text in remarks.fillna("")
    for character in text
    if ord(character) > 127
)

print("Number of distinct non-ASCII characters:", len(unicode_characters))
print("\nMost common non-ASCII characters:")

for character, count in unicode_characters.most_common(30):
    print(repr(character), ord(character), count)

Number of distinct non-ASCII characters: 22

Most common non-ASCII characters:
'’' 8217 388
'—' 8212 362
'\u200b' 8203 120
'\u200c' 8204 80
'•' 8226 47
'–' 8211 34
'\xa0' 160 22
'”' 8221 15
'é' 233 12
'±' 177 11
'“' 8220 11
'®' 174 9
'½' 189 4
'°' 176 2
'â' 226 2
'á' 225 1
'…' 8230 1
'¼' 188 1
'ñ' 241 1
'Ã' 195 1
'©' 169 1
'‘' 8216 1


In [9]:
suspicious_characters = ["â", "Ã", "\u200b", "\u200c"]

for character in suspicious_characters:
    matches = df[
        remarks.str.contains(
            character,
            regex=False,
            na=False
        )
    ]["remarks"]

    print(f"\nCHARACTER: {repr(character)}")
    print("MATCHING REMARKS:", len(matches))

    for text in matches.head(3):
        position = text.find(character)
        start = max(0, position - 80)
        end = min(len(text), position + 120)
        print(repr(text[start:end]))


CHARACTER: 'â'
MATCHING REMARKS: 1
" opportunity to own one of La Jolla's most coveted newly remodeled oceanfront châteaux. This exceptional residence showcases breathtaking, unobstructed panoramic views of the Pacific Ocean and iconic "

CHARACTER: 'Ã'
MATCHING REMARKS: 1
"e location just steps from Main Street Cupertino's vibrant dining, shopping, cafÃ©s, and entertainment, with easy access to Apple, major tech campuses, Highway 280, and top-rated Cupertino schools. A "

CHARACTER: '\u200b'
MATCHING REMARKS: 5
' downsizers or Investors.  Average rental for a 2 bedroom Condo in Pleasanton is\u200b\u200b\u200c\u200b\u200b\u200b\u200b\u200c\u200b\u200b\u200c\u200c\u200b\u200b\u200c\u200b\u200b\u200b\u200c\u200c\u200b\u200b\u200b\u200c\u200b\u200b\u200c\u200c\u200b\u200c\u200c\u200c\u200b\u200c\u200b\u200b\u200b\u200c\u200c\u200b $2800.00'
'BILITIES - Very hard to find homes in this area at this price point! Take a look\u200b\u200b\u200c\u200b\u200b\u200b\u200b\u200c\u200b\u200b\u200c\u

In [10]:
abbreviation_patterns = {
    "br": r"\bbr\b",
    "bd": r"\bbd\b",
    "ba": r"\bba\b",
    "mbr": r"\bmbr\b",
    "sqft": r"\bsqft\b",
    "sq ft": r"\bsq\.?\s*ft\.?\b",
    "w/": r"(?<!\w)w/",
    "w/o": r"(?<!\w)w/o",
    "a/c": r"\ba/c\b",
    "ac": r"\bac\b",
    "hvac": r"\bhvac\b",
    "hoa": r"\bhoa\b",
    "adu": r"\badu\b",
    "ev": r"\bev\b",
    "rv": r"\brv\b"
}

abbreviation_counts = {}

for abbreviation, pattern in abbreviation_patterns.items():
    abbreviation_counts[abbreviation] = remarks.str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    ).sum()

for abbreviation, count in sorted(
    abbreviation_counts.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{abbreviation}: {count}")

sq ft: 172
hoa: 124
rv: 80
hvac: 65
adu: 60
ev: 54
a/c: 51
ac: 36
sqft: 24
w/: 23
bd: 3
ba: 3
w/o: 3
br: 0
mbr: 0


In [ ]:
Handle missing input
→ normalize Unicode
→ remove HTML
→ normalize prices
→ normalize measurements
→ expand abbreviations
→ normalize whitespace